# Project 12: Oil Well Selection — Profitability and Risk Analysis

## Project Overview

The oil company OilyGiant wants to identify the most profitable region in which to develop 200 new oil wells. To support this decision, we will analyze data from three regions and build a linear regression model to predict the volume of reserves in new wells.

The objective is to select the region with the best expected financial outcome while taking both profitability and the risk of losses into account.

## Project Objectives

- Prepare and analyze the datasets for all three regions.
- Train a linear regression model to predict oil reserves.
- Evaluate each model using RMSE and the mean predicted reserve volume.
- Select the 200 wells with the highest predicted reserves in each region.
- Calculate the potential profit for each region.
- Use bootstrapping to estimate mean profit, the 95% confidence interval, and the probability of a loss.
- Recommend a region with a loss risk below 2.5% and the highest mean profit among the regions that meet this criterion.

## Business Conditions

The budget for developing 200 wells is $100 million. Each unit of product represents 1,000 barrels and generates $4,500 in revenue. The risk analysis will use 1,000 bootstrap samples.

## 1. Data Preparation

We will load the three datasets, one for each region. First, we will inspect their structure and available data to make sure they are suitable for analysis and model training.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

### Load the Data

We will load the data for the three regions from their CSV files and store them in separate variables for further analysis.

In [2]:
data_0 = pd.read_csv('/datasets/geo_data_0.csv')
data_1 = pd.read_csv('/datasets/geo_data_1.csv')
data_2 = pd.read_csv('/datasets/geo_data_2.csv')

display(data_0.head())
display(data_1.head())
display(data_2.head())

,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305


,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746


### Check for Missing Values and Duplicate Rows

We will check whether any of the three datasets contain missing values or duplicate rows. These data quality issues should be identified before training the models.

In [3]:
print('Valores ausentes:')
print('Región 0:', data_0.isna().sum().sum())
print('Región 1:', data_1.isna().sum().sum())
print('Región 2:', data_2.isna().sum().sum())

print('\nFilas duplicadas:')
print('Región 0:', data_0.duplicated().sum())
print('Región 1:', data_1.duplicated().sum())
print('Región 2:', data_2.duplicated().sum())

Valores ausentes:
Región 0: 0
Región 1: 0
Región 2: 0

Filas duplicadas:
Región 0: 0
Región 1: 0
Región 2: 0


### Explore the Data Characteristics

We will use descriptive statistics to understand the distributions of the features and the reserve volume in each region. We are particularly interested in the mean of `product`, which represents the average reserve volume per well.

In [4]:
display(data_0.describe())
display(data_1.describe())
display(data_2.describe())

,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.500419,0.250143,2.502647,92.500000
std,0.871832,0.504433,3.248248,44.288691
min,-1.408605,-0.848218,-12.088328,0.000000
25%,-0.072580,-0.200881,0.287748,56.497507
50%,0.502360,0.250252,2.515969,91.849972
75%,1.073581,0.700646,4.715088,128.564089
max,2.362331,1.343769,16.003790,185.364347


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,1.141296,-4.796579,2.494541,68.825000
std,8.965932,5.119872,1.703572,45.944423
min,-31.609576,-26.358598,-0.018144,0.000000
25%,-6.298551,-8.267985,1.000021,26.953261
50%,1.153055,-4.813172,2.011479,57.085625
75%,8.621015,-1.332816,3.999904,107.813044
max,29.421755,18.734063,5.019721,137.945408


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.002023,-0.002081,2.495128,95.000000
std,1.732045,1.730417,3.473445,44.749921
min,-8.760004,-7.084020,-11.970335,0.000000
25%,-1.162288,-1.174820,0.130359,59.450441
50%,0.009424,-0.009482,2.484236,94.925613
75%,1.158535,1.163678,4.858794,130.595027
max,7.238262,7.844801,16.739402,190.029838


### Data Preparation Summary

All three datasets contain 100,000 records and the same five variables. No missing values or duplicate rows were found.

The variables `f0`, `f1`, and `f2` are numerical and will be used as model features. The `product` variable is the target we want to predict.

The mean reserve volume is 92.50 units in Region 0, 68.83 units in Region 1, and 95.00 units in Region 2. Region 2 has the highest overall mean, followed by Region 0 and Region 1.

However, the average reserve volume alone is not enough to determine which region is the most profitable. In the next step, we will train and evaluate a linear regression model for each region.

## 2. Model Training and Evaluation

We will train a linear regression model for each region to predict reserve volume.

The data will be split into a 75% training set and a 25% validation set. The `id` column will not be used as a feature because it only identifies each well.

To evaluate the models, we will calculate the mean predicted reserve volume and RMSE.

In [5]:
def train_and_evaluate(data):
    features = data.drop(['id', 'product'], axis=1)
    target = data['product']

    features_train, features_valid, target_train, target_valid = train_test_split(
        features,
        target,
        test_size=0.25,
        random_state=12345
    )

    model = LinearRegression()
    model.fit(features_train, target_train)

    predictions = model.predict(features_valid)

    mean_prediction = predictions.mean()
    rmse = mean_squared_error(target_valid, predictions) ** 0.5

    return target_valid, predictions, mean_prediction, rmse

### Train and Evaluate the Models for All Three Regions

We will apply the function to each region and store the actual target values, predictions, mean predicted reserve volume, and RMSE.

These results will allow us to compare model accuracy and the reserve volumes predicted for each region.

In [6]:
target_valid_0, predictions_0, mean_prediction_0, rmse_0 = train_and_evaluate(data_0)
target_valid_1, predictions_1, mean_prediction_1, rmse_1 = train_and_evaluate(data_1)
target_valid_2, predictions_2, mean_prediction_2, rmse_2 = train_and_evaluate(data_2)

for region, mean_prediction, rmse in [
    (0, mean_prediction_0, rmse_0),
    (1, mean_prediction_1, rmse_1),
    (2, mean_prediction_2, rmse_2)
]:
    print(f'Región {region}')
    print(f'Volumen medio de reservas predicho: {mean_prediction:.2f}')
    print(f'RMSE: {rmse:.2f}')
    print()

Región 0
Volumen medio de reservas predicho: 92.59
RMSE: 37.58

Región 1
Volumen medio de reservas predicho: 68.73
RMSE: 0.89

Región 2
Volumen medio de reservas predicho: 94.97
RMSE: 40.03



### Analysis of the Model Results

Region 2 has the highest mean predicted reserve volume, at 94.97 units per well, followed by Region 0 at 92.59 units. Region 1 has the lowest mean predicted volume, at 68.73 units.

However, RMSE reveals an important difference between the regions. The model for Region 1 has an RMSE of just 0.89, while the models for Regions 0 and 2 have RMSE values of 37.58 and 40.03, respectively.

This indicates that the predictions for Region 1 are much more precise. Nevertheless, we cannot choose a region based only on mean reserve volume or RMSE. In the following steps, we will examine profitability and the risk of losses.

## 3. Preparing the Profit Calculation

Before calculating profit, we will define the project's financial assumptions as separate variables. This makes the calculations clear and ensures that the same values are used consistently throughout the analysis.

### Define the Required Variables

The available budget for developing 200 wells is $100 million. Each unit of product represents 1,000 barrels and generates $4,500 in revenue.

We will also define that the 200 wells with the highest predicted reserves will be selected.

In [7]:
BUDGET = 100_000_000
N_WELLS = 200
N_SAMPLES = 500
REVENUE_PER_UNIT = 4_500

### Calculate the Minimum Required Reserve Volume

We will calculate the average number of product units each well must produce to cover the total budget of $100 million across 200 wells.

This minimum volume will serve as a reference when comparing the average reserves in the three regions.

In [8]:
min_volume_per_well = BUDGET / N_WELLS / REVENUE_PER_UNIT

print(f'Volumen mínimo por pozo: {min_volume_per_well:.2f} unidades')

Volumen mínimo por pozo: 111.11 unidades


### Compare the Minimum Required Volume with the Regional Averages

We will compare the minimum reserve volume required to cover costs with the mean reserve volume in each region.

This comparison is only an initial reference. To calculate profit, we will use the 200 wells with the highest predictions in each sample.

In [9]:
mean_product_0 = data_0['product'].mean()
mean_product_1 = data_1['product'].mean()
mean_product_2 = data_2['product'].mean()

print(f'Región 0: {mean_product_0:.2f} unidades por pozo')
print(f'Región 1: {mean_product_1:.2f} unidades por pozo')
print(f'Región 2: {mean_product_2:.2f} unidades por pozo')

print(f'\nVolumen mínimo necesario: {min_volume_per_well:.2f} unidades por pozo')

Región 0: 92.50 unidades por pozo
Región 1: 68.83 unidades por pozo
Región 2: 95.00 unidades por pozo

Volumen mínimo necesario: 111.11 unidades por pozo


### Conclusion of the Initial Profit Assessment

The mean reserve volume in all three regions is below the 111.11 units per well required to cover the budget.

Region 2 has the highest overall mean, at 95.00 units, followed by Region 0 at 92.50 units. Region 1 has the lowest average, at 68.83 units.

However, these values represent all wells in each region. The development plan will select the 200 wells with the highest predicted reserves. Therefore, the average reserve volume of the selected wells may be higher than the overall regional average.

## 4. Profit Calculation

In this step, we will select the 200 wells with the highest predicted reserve volumes in each region and calculate their potential profit.

The profit calculation will use the actual reserves of the selected wells, not the predictions. This allows us to estimate the potential profit resulting from the model-based selection.

In [10]:
def calculate_profit(target, predictions):
    top_200 = np.argsort(predictions)[-N_WELLS:]
    total_volume = target.iloc[top_200].sum()
    revenue = total_volume * REVENUE_PER_UNIT
    profit = revenue - BUDGET
    
    return total_volume, profit

### Select the 200 Best Wells in Each Region

We will apply the model to each region and select the 200 wells with the highest predicted reserve volumes.

We will then calculate the actual total reserve volume of these wells and their potential profit.

In [11]:
top_200_0 = pd.Series(predictions_0, index=target_valid_0.index).nlargest(N_WELLS)
top_200_1 = pd.Series(predictions_1, index=target_valid_1.index).nlargest(N_WELLS)
top_200_2 = pd.Series(predictions_2, index=target_valid_2.index).nlargest(N_WELLS)

volume_0, profit_0 = calculate_profit(target_valid_0, predictions_0)
volume_1, profit_1 = calculate_profit(target_valid_1, predictions_1)
volume_2, profit_2 = calculate_profit(target_valid_2, predictions_2)

print(f'Región 0: volumen total = {volume_0:.2f}, beneficio = ${profit_0:,.2f}')
print(f'Región 1: volumen total = {volume_1:.2f}, beneficio = ${profit_1:,.2f}')
print(f'Región 2: volumen total = {volume_2:.2f}, beneficio = ${profit_2:,.2f}')

Región 0: volumen total = 29601.84, beneficio = $33,208,260.43
Región 1: volumen total = 27589.08, beneficio = $24,150,866.97
Región 2: volumen total = 28245.22, beneficio = $27,103,499.64


### Compare the Potential Profit

Region 0 has the highest total reserve volume among the 200 selected wells and also the highest potential profit, at approximately $33.21 million.

Region 2 ranks second, with a potential profit of approximately $27.10 million. Region 1 has the lowest potential profit, at approximately $24.15 million.

Based on this initial analysis, Region 0 appears to be the best choice. However, this comparison is based on a single validation sample and does not yet account for uncertainty or the risk of losses. The final decision will therefore be made after the bootstrap analysis.

## 5. Risk and Profit Analysis

We will use bootstrapping to evaluate the uncertainty of the profit estimates and the risk of losses.

For each region, we will generate 1,000 random samples of 500 wells with replacement. In each sample, we will select the 200 wells with the highest predicted reserves and calculate the resulting profit using their actual reserve volumes.

From the 1,000 profit estimates, we will calculate the mean profit, the 95% confidence interval, and the probability of making a loss.

### Create the Bootstrap Function

We will create a function that generates 1,000 samples of 500 wells with replacement. For each sample, the 200 wells with the highest predicted reserves are selected and the corresponding profit is calculated.

The function returns the distribution of profit estimates from all 1,000 samples.

In [12]:
def bootstrap_profit(target, predictions, n_samples=1000, sample_size=N_SAMPLES, random_state=12345):
    rng = np.random.RandomState(random_state)
    profits = []

    for _ in range(n_samples):
        sample_indices = rng.choice(len(target), size=sample_size, replace=True)

        sample_target = target.iloc[sample_indices]
        sample_predictions = predictions[sample_indices]

        _, profit = calculate_profit(sample_target, sample_predictions)
        profits.append(profit)

    return pd.Series(profits)

### Bootstrap Analysis for Region 0

We will apply the bootstrap function to Region 0. We will generate 1,000 samples of 500 wells with replacement and calculate the profit from selecting the 200 wells with the highest predictions in each sample.

In [13]:
profits_0 = bootstrap_profit(
    target_valid_0,
    predictions_0
)

print(f'Número de muestras: {len(profits_0)}')
print(f'Beneficio medio: ${profits_0.mean():,.2f}')

Número de muestras: 1000
Beneficio medio: $3,961,649.85


### Confidence Interval and Loss Risk for Region 0

We will calculate the 95% confidence interval for profit and the probability of making a loss.

The loss probability will then be compared with the maximum permitted threshold of 2.5%.

In [14]:
lower_0 = profits_0.quantile(0.025)
upper_0 = profits_0.quantile(0.975)
risk_0 = (profits_0 < 0).mean()

print(f'Intervalo de confianza del 95 %: ${lower_0:,.2f} - ${upper_0:,.2f}')
print(f'Probabilidad de pérdida: {risk_0:.2%}')

Intervalo de confianza del 95 %: $-1,112,155.46 - $9,097,669.42
Probabilidad de pérdida: 6.90%


### Bootstrap Analysis for Regions 1 and 2

We will apply the same procedure to Regions 1 and 2. For each region, we will generate 1,000 samples of 500 wells and calculate the mean profit, the 95% confidence interval, and the probability of a loss.

In [15]:
profits_1 = bootstrap_profit(
    target_valid_1,
    predictions_1
)

profits_2 = bootstrap_profit(
    target_valid_2,
    predictions_2
)

for region, profits in [(1, profits_1), (2, profits_2)]:
    lower = profits.quantile(0.025)
    upper = profits.quantile(0.975)
    risk = (profits < 0).mean()

    print(f'Región {region}')
    print(f'Beneficio medio: ${profits.mean():,.2f}')
    print(f'Intervalo de confianza del 95 %: ${lower:,.2f} - ${upper:,.2f}')
    print(f'Probabilidad de pérdida: {risk:.2%}')
    print()

Región 1
Beneficio medio: $4,560,451.06
Intervalo de confianza del 95 %: $338,205.09 - $8,522,894.54
Probabilidad de pérdida: 1.50%

Región 2
Beneficio medio: $4,044,038.67
Intervalo de confianza del 95 %: $-1,633,504.13 - $9,503,595.75
Probabilidad de pérdida: 7.60%



### Final Comparison of the Regions

The bootstrap results reveal important differences between the three regions.

Region 0 has a mean profit of approximately $3.96 million, but its probability of a loss is 6.90%, which is above the permitted threshold of 2.5%.

Region 2 has a mean profit of approximately $4.04 million, but it also has a high loss risk of 7.60%.

Region 1 has the highest mean profit, at approximately $4.56 million, and a loss probability of only 1.50%. Its 95% confidence interval also remains above zero.

Therefore, Region 1 is the only region that meets the requirement of keeping the probability of a loss below 2.5%, while also providing the highest mean profit among the eligible regions.

## 6. Final Conclusion and Recommendation

In the initial analysis, Region 0 appeared to be the best option because the 200 selected wells generated the highest potential profit.

However, the bootstrap analysis shows that Region 0 has a 6.90% probability of a loss, which is above the permitted threshold of 2.5%. Region 2 also exceeds this limit, with a loss risk of 7.60%.

Region 1 is the only region that meets the risk requirement, with a loss probability of 1.50%. It also has the highest mean profit, at approximately $4.56 million.

We therefore recommend developing the 200 wells in Region 1. Although its initial potential profit was lower than that of Region 0, the risk analysis shows that Region 1 offers the best balance between profitability and risk.